# Lab: LangGraph Advanced — Production Patterns for NovaPay

## Learning Objectives

By the end of this lab, you will:

- Build parallel branch execution (fan-out/fan-in) for multi-check fraud analysis
- Create subgraphs for complex investigation workflows
- Implement human-in-the-loop with interrupt/resume
- Add persistent checkpointing (survive restarts)
- Stream node outputs for real-time UI
- Handle errors with retry and graceful degradation
- Compare LangGraph vs Strands for production deployments

**Scenario:** NovaPay's fraud system needs a stateful investigation pipeline that runs fraud + compliance checks in parallel, pauses for human approval before blocking cards, and can resume after system restarts.

**Prerequisites:** Basic LangGraph knowledge (nodes, edges, conditional routing)

**Time:** ~75 minutes | **Level:** Advanced

In [ ]:
%pip install -q -r requirements.txt

## Section 1: Quick Recap — LangGraph Fundamentals

LangGraph models agent workflows as directed graphs with:

- **Nodes** = functions that transform state
- **Edges** = transitions between nodes
- **Conditional edges** = routing based on state
- **State** = TypedDict that flows through the graph

**Key insight:** Unlike LangChain's AgentExecutor (linear ReAct loop), LangGraph gives you explicit control over the execution topology.

In [ ]:
import boto3
import json
import time
import datetime
import asyncio
from datetime import datetime
from typing import Dict, List, Optional, Any, TypedDict, Annotated, Literal
from IPython.display import HTML, display

display(HTML('<h2 style="color: #9b59b6;">🔀 NovaPay LangGraph Advanced Lab</h2>'))
display(HTML('<p style="color: #7f8c8d;">Parallel execution, HITL, checkpointing, streaming</p>'))

In [ ]:
from langchain_aws import ChatBedrock
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages

# Model setup
MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
REGION = "us-east-1"

llm = ChatBedrock(
    model_id=MODEL_ID,
    client=boto3.client("bedrock-runtime", region_name=REGION),
    model_kwargs={"temperature": 0.0, "max_tokens": 2048},
)

print(f"✅ Model ready: {MODEL_ID}")

In [ ]:
# ==========================================
# NovaPay Shared Data
# ==========================================

NOVAPAY_ACCOUNTS = {
    "CUST-1001": {
        "name": "Amara Okafor",
        "balance": 15420.50,
        "currency": "USD",
        "account_type": "premium",
        "status": "active",
        "risk_score": 72,
        "last_login": "2024-01-15T09:30:00Z",
        "login_location": "Lagos, Nigeria",
        "kyc_level": "enhanced",
        "sanctions_clear": True,
    }
}

NOVAPAY_TRANSACTIONS = {
    "TXN-101": {
        "customer_id": "CUST-1001",
        "amount": 4999.99,
        "currency": "USD",
        "merchant": "CryptoExchange_EU",
        "category": "crypto",
        "timestamp": "2024-01-15T03:22:00Z",
        "location": "Tallinn, Estonia",
        "device": "new_device_fingerprint_x9f2",
        "velocity_flag": True,
        "status": "pending_review",
    },
    "TXN-102": {
        "customer_id": "CUST-1001",
        "amount": 3200.00,
        "currency": "USD",
        "merchant": "CryptoExchange_EU",
        "category": "crypto",
        "timestamp": "2024-01-15T03:45:00Z",
        "location": "Tallinn, Estonia",
        "device": "new_device_fingerprint_x9f2",
        "velocity_flag": True,
        "status": "pending_review",
    },
    "TXN-103": {
        "customer_id": "CUST-1001",
        "amount": 150.00,
        "currency": "USD",
        "merchant": "Netflix",
        "category": "entertainment",
        "timestamp": "2024-01-14T18:00:00Z",
        "location": "Lagos, Nigeria",
        "device": "known_device_amara_iphone",
        "velocity_flag": False,
        "status": "completed",
    },
}

print(f"✅ Data loaded: {len(NOVAPAY_ACCOUNTS)} accounts, {len(NOVAPAY_TRANSACTIONS)} transactions")

## Section 2: Parallel Branch Execution (Fan-Out / Fan-In)

**Problem:** NovaPay needs to run fraud analysis AND compliance checks simultaneously. Running them sequentially doubles latency. With LangGraph, we fan-out to parallel branches, and fan-in to synthesize results.

```
intake -> fraud_check      \
                             -> synthesize -> decision
       -> compliance_check /
```

In [ ]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Annotated
import operator

# State with parallel results
class InvestigationState(TypedDict):
    transaction_id: str
    customer_id: str
    # Parallel branch results (use Annotated + operator.add to merge lists)
    fraud_result: Optional[Dict[str, Any]]
    compliance_result: Optional[Dict[str, Any]]
    final_decision: Optional[str]
    risk_score: int
    evidence: Annotated[List[str], operator.add]  # Accumulates across nodes
    error_log: Annotated[List[str], operator.add]


def intake_node(state: InvestigationState) -> dict:
    """Initial triage - validate inputs, load context."""
    txn_id = state["transaction_id"]
    if txn_id not in NOVAPAY_TRANSACTIONS:
        return {"error_log": [f"Transaction {txn_id} not found"], "risk_score": 0}

    txn = NOVAPAY_TRANSACTIONS[txn_id]
    return {
        "customer_id": txn["customer_id"],
        "evidence": [f"Intake: Transaction {txn_id} for ${txn['amount']} to {txn['merchant']}"],
    }


def fraud_check_node(state: InvestigationState) -> dict:
    """Fraud analysis branch - runs ML signals."""
    txn = NOVAPAY_TRANSACTIONS.get(state["transaction_id"], {})
    signals = []
    score = 0

    if txn.get("velocity_flag"):
        signals.append("velocity_anomaly")
        score += 30
    if "new_device" in txn.get("device", ""):
        signals.append("new_device")
        score += 20
    if txn.get("category") in ["crypto", "gambling"]:
        signals.append("high_risk_category")
        score += 15

    customer = NOVAPAY_ACCOUNTS.get(state.get("customer_id", ""), {})
    if customer and txn.get("location") != customer.get("login_location"):
        signals.append("geo_mismatch")
        score += 25

    time.sleep(0.1)  # Simulate ML model latency

    return {
        "fraud_result": {"score": score, "signals": signals},
        "risk_score": score,
        "evidence": [f"Fraud check: score={score}, signals={signals}"],
    }


def compliance_check_node(state: InvestigationState) -> dict:
    """Compliance branch - AML/KYC/sanctions checks."""
    customer = NOVAPAY_ACCOUNTS.get(state.get("customer_id", ""), {})
    txn = NOVAPAY_TRANSACTIONS.get(state["transaction_id"], {})

    issues = []

    # AML threshold check
    if txn.get("amount", 0) > 3000:
        issues.append("exceeds_aml_reporting_threshold")

    # Cross-border to high-risk jurisdiction
    high_risk_jurisdictions = ["Estonia", "Malta", "Cayman Islands"]
    if any(j in txn.get("location", "") for j in high_risk_jurisdictions):
        issues.append("high_risk_jurisdiction")

    # KYC level vs transaction size
    if txn.get("amount", 0) > 5000 and customer.get("kyc_level") != "enhanced":
        issues.append("insufficient_kyc_for_amount")

    time.sleep(0.1)  # Simulate API call

    return {
        "compliance_result": {
            "issues": issues,
            "sanctions_clear": customer.get("sanctions_clear", False),
            "kyc_level": customer.get("kyc_level", "basic"),
        },
        "evidence": [f"Compliance: issues={issues}, sanctions_clear={customer.get('sanctions_clear')}"],
    }


def synthesize_node(state: InvestigationState) -> dict:
    """Fan-in: combine fraud + compliance results into final decision."""
    fraud = state.get("fraud_result", {}) or {}
    compliance = state.get("compliance_result", {}) or {}

    fraud_score = fraud.get("score", 0)
    compliance_issues = compliance.get("issues", [])

    # Decision logic
    if fraud_score >= 80 or (fraud_score >= 60 and len(compliance_issues) > 1):
        decision = "BLOCK"
    elif fraud_score >= 50 or len(compliance_issues) > 0:
        decision = "HUMAN_REVIEW"
    else:
        decision = "APPROVE"

    return {
        "final_decision": decision,
        "evidence": [f"Decision: {decision} (fraud={fraud_score}, compliance_issues={len(compliance_issues)})"],
    }


# Build the parallel graph
parallel_graph = StateGraph(InvestigationState)

parallel_graph.add_node("intake", intake_node)
parallel_graph.add_node("fraud_check", fraud_check_node)
parallel_graph.add_node("compliance_check", compliance_check_node)
parallel_graph.add_node("synthesize", synthesize_node)

# Edges: intake fans out to both checks, both fan into synthesize
parallel_graph.add_edge(START, "intake")
parallel_graph.add_edge("intake", "fraud_check")
parallel_graph.add_edge("intake", "compliance_check")
parallel_graph.add_edge("fraud_check", "synthesize")
parallel_graph.add_edge("compliance_check", "synthesize")
parallel_graph.add_edge("synthesize", END)

app = parallel_graph.compile()
print("✅ Parallel investigation graph compiled")
print("  Nodes: intake → [fraud_check ┃ compliance_check] → synthesize")

In [ ]:
# Run the parallel investigation
start_time = time.time()

result = app.invoke({
    "transaction_id": "TXN-101",
    "customer_id": "",
    "fraud_result": None,
    "compliance_result": None,
    "final_decision": None,
    "risk_score": 0,
    "evidence": [],
    "error_log": [],
})

elapsed = time.time() - start_time

print(f"⏱️ Total time: {elapsed:.2f}s (parallel branches save ~50% vs sequential)")
print(f"\n📋 Investigation Results:")
print(f"  Decision: {result['final_decision']}")
print(f"  Risk Score: {result['risk_score']}")
print(f"  Fraud Signals: {result['fraud_result']['signals']}")
print(f"  Compliance Issues: {result['compliance_result']['issues']}")
print(f"\n📝 Evidence Trail:")
for e in result['evidence']:
    print(f"  - {e}")

## Section 3: Subgraphs — Complex Investigations as Modular Components

When investigations get complex, you encapsulate logic in **subgraphs**. NovaPay uses this for "deep investigation" — called only when the initial fraud score is above 60. This is like microservices for agent logic — test and develop independently.

In [ ]:
# Deep investigation subgraph - called when fraud_score > 60
class DeepInvestigationState(TypedDict):
    transaction_id: str
    customer_id: str
    related_transactions: List[Dict]
    network_analysis: Optional[Dict]
    final_risk: int
    evidence: Annotated[List[str], operator.add]


def find_related_transactions(state: DeepInvestigationState) -> dict:
    """Find other transactions from same customer in the same time window."""
    customer_id = state["customer_id"]
    related = [
        {"id": tid, **tdata}
        for tid, tdata in NOVAPAY_TRANSACTIONS.items()
        if tdata["customer_id"] == customer_id
    ]
    return {
        "related_transactions": related,
        "evidence": [f"Found {len(related)} related transactions for {customer_id}"],
    }


def network_analysis_node(state: DeepInvestigationState) -> dict:
    """Analyze transaction network for structuring patterns."""
    related = state.get("related_transactions", [])

    # Check for structuring (splitting to avoid thresholds)
    amounts = [t["amount"] for t in related]
    total = sum(amounts)

    # Structuring detection: multiple transactions just under $5000
    near_threshold = [a for a in amounts if 3000 <= a <= 5000]
    structuring_flag = len(near_threshold) >= 2

    # Same merchant pattern
    merchants = [t.get("merchant") for t in related]
    same_merchant = len(set(merchants)) < len(merchants)

    return {
        "network_analysis": {
            "total_amount": total,
            "structuring_detected": structuring_flag,
            "same_merchant_pattern": same_merchant,
            "transaction_count": len(related),
        },
        "evidence": [f"Network: total=${total:.2f}, structuring={structuring_flag}, same_merchant={same_merchant}"],
    }


def calculate_deep_risk(state: DeepInvestigationState) -> dict:
    """Calculate final risk from deep investigation."""
    network = state.get("network_analysis", {}) or {}
    risk = 50  # Base risk (already flagged to get here)

    if network.get("structuring_detected"):
        risk += 30
    if network.get("same_merchant_pattern"):
        risk += 10
    if network.get("total_amount", 0) > 10000:
        risk += 10

    return {
        "final_risk": min(risk, 100),
        "evidence": [f"Deep investigation risk: {min(risk, 100)}"],
    }


# Build subgraph
deep_investigation = StateGraph(DeepInvestigationState)
deep_investigation.add_node("find_related", find_related_transactions)
deep_investigation.add_node("network_analysis", network_analysis_node)
deep_investigation.add_node("calculate_risk", calculate_deep_risk)

deep_investigation.add_edge(START, "find_related")
deep_investigation.add_edge("find_related", "network_analysis")
deep_investigation.add_edge("network_analysis", "calculate_risk")
deep_investigation.add_edge("calculate_risk", END)

deep_app = deep_investigation.compile()

# Test subgraph independently
result = deep_app.invoke({
    "transaction_id": "TXN-101",
    "customer_id": "CUST-1001",
    "related_transactions": [],
    "network_analysis": None,
    "final_risk": 0,
    "evidence": [],
})

print("✅ Subgraph test:")
print(f"  Final risk: {result['final_risk']}")
print(f"  Network: {result['network_analysis']}")
print(f"  Evidence: {result['evidence']}")

## Section 4: Human-in-the-Loop with Interrupt

**Critical pattern for NovaPay:** Before freezing a customer's card, the system MUST pause and get human approval. LangGraph's `interrupt_before` mechanism:

1. Graph executes until it reaches the interrupt node
2. State is persisted (checkpointed)
3. System waits for human input
4. On approval, graph resumes from the interrupt point

This is **regulatory required** for card freezes exceeding $1000.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

class HITLState(TypedDict):
    transaction_id: str
    risk_score: int
    recommendation: str
    human_decision: Optional[str]
    action_taken: Optional[str]
    evidence: Annotated[List[str], operator.add]


def analyze_risk(state: HITLState) -> dict:
    """Automated risk analysis."""
    txn = NOVAPAY_TRANSACTIONS.get(state["transaction_id"], {})
    score = 85  # High risk for demo
    recommendation = "FREEZE_CARD" if score >= 80 else "MONITOR"
    return {
        "risk_score": score,
        "recommendation": recommendation,
        "evidence": [f"Risk analysis complete: score={score}, recommend={recommendation}"],
    }


def freeze_card(state: HITLState) -> dict:
    """Execute card freeze - only runs after human approval."""
    # In production, this calls the card management API
    return {
        "action_taken": "CARD_FROZEN",
        "evidence": [f"Card frozen for transaction {state['transaction_id']} after human approval"],
    }


def notify_customer(state: HITLState) -> dict:
    """Notify customer of the action taken."""
    action = state.get("action_taken", "NONE")
    return {
        "evidence": [f"Customer notified: action={action}"],
    }


# Build HITL graph with interrupt
hitl_graph = StateGraph(HITLState)
hitl_graph.add_node("analyze", analyze_risk)
hitl_graph.add_node("freeze_card", freeze_card)
hitl_graph.add_node("notify", notify_customer)

hitl_graph.add_edge(START, "analyze")
hitl_graph.add_edge("analyze", "freeze_card")
hitl_graph.add_edge("freeze_card", "notify")
hitl_graph.add_edge("notify", END)

# Compile with interrupt BEFORE freeze_card
checkpointer = MemorySaver()
hitl_app = hitl_graph.compile(
    checkpointer=checkpointer,
    interrupt_before=["freeze_card"],  # Pause here for human approval
)

print("✅ HITL graph compiled with interrupt before freeze_card")
print("  Flow: analyze → [INTERRUPT] → freeze_card → notify")

In [ ]:
# Step 1: Run until interrupt
config = {"configurable": {"thread_id": "investigation-001"}}

initial_state = {
    "transaction_id": "TXN-101",
    "risk_score": 0,
    "recommendation": "",
    "human_decision": None,
    "action_taken": None,
    "evidence": [],
}

# This will stop BEFORE freeze_card
result = hitl_app.invoke(initial_state, config)

print(f"🔴 Graph paused at interrupt point")
print(f"  Risk Score: {result['risk_score']}")
print(f"  Recommendation: {result['recommendation']}")
print(f"  Evidence so far: {result['evidence']}")
print(f"\nℹ️  Waiting for human decision...")
print(f"  (In production: Slack notification sent to fraud ops team)")

In [ ]:
# Step 2: Human approves - resume the graph
# In production, this comes from a Slack button, web UI, or API call

print("👤 Human analyst approves: FREEZE_CARD")
print("  (Simulating approval after reviewing evidence)\n")

# Resume from checkpoint - the graph continues from where it paused
result = hitl_app.invoke(
    {"human_decision": "APPROVED"},  # Update state with human input
    config,  # Same thread_id - resumes from checkpoint
)

print(f"✅ Graph resumed and completed")
print(f"  Action taken: {result['action_taken']}")
print(f"  Full evidence trail:")
for e in result['evidence']:
    print(f"  - {e}")

## Section 5: Persistent Checkpointing with SQLite

In-memory checkpointing (MemorySaver) is fine for development, but production needs persistence. LangGraph supports SQLite, PostgreSQL, and custom backends.

**Why NovaPay needs this:**

- Investigations can span hours (waiting for human review)
- Server restarts shouldn't lose investigation state
- Compliance requires full audit trail of every state transition

In [ ]:
# SQLite checkpointing for persistence
# In production: PostgreSQL or DynamoDB for scalability
import sqlite3
import os

# Demonstrate the pattern (using MemorySaver as SQLite requires langgraph-checkpoint-sqlite)
# Production code would use:
# from langgraph.checkpoint.sqlite import SqliteSaver
# checkpointer = SqliteSaver.from_conn_string("sqlite:///novapay_investigations.db")

# For this demo, we show the checkpoint data structure
print("💾 Checkpointing Architecture:")
print()
print("  Development:  MemorySaver (in-process, lost on restart)")
print("  Testing:      SqliteSaver (file-based, survives restarts)")
print("  Production:   PostgresSaver (scalable, shared across instances)")
print("  AWS Native:   DynamoDB custom backend (serverless, auto-scaling)")
print()
print("Each checkpoint stores:")
print("  - Full state snapshot")
print("  - Thread ID (investigation ID)")
print("  - Timestamp")
print("  - Parent checkpoint (for state history)")
print("  - Metadata (which node was next)")
print()

# Show what's in our in-memory checkpoint
checkpoint_state = hitl_app.get_state(config)
print(f"Current checkpoint state for thread 'investigation-001':")
print(f"  Next node: {checkpoint_state.next}")
print(f"  Values keys: {list(checkpoint_state.values.keys())}")

## Section 6: Streaming with Events — Real-Time UI Updates

For NovaPay's investigation dashboard, analysts need to see progress in real-time:

- Which checks are running
- Results as they come in
- Time spent per node

LangGraph's `stream()` and `astream_events()` enable this.

In [ ]:
# Streaming node-by-node output
print("📡 Streaming investigation (node-by-node):")
print("=" * 50)

stream_state = {
    "transaction_id": "TXN-102",
    "customer_id": "",
    "fraud_result": None,
    "compliance_result": None,
    "final_decision": None,
    "risk_score": 0,
    "evidence": [],
    "error_log": [],
}

for event in app.stream(stream_state, stream_mode="updates"):
    for node_name, node_output in event.items():
        timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        print(f"  [{timestamp}] Node '{node_name}' completed:")
        # Show key outputs without flooding
        if "evidence" in node_output:
            for e in node_output["evidence"]:
                print(f"      → {e}")
        if "final_decision" in node_output:
            print(f"      🎯 DECISION: {node_output['final_decision']}")

print("\n✅ Stream complete - all nodes executed")
print("  In production: each event would be sent via WebSocket to the analyst UI")

## Section 7: Error Recovery — Retry and Graceful Degradation

Production systems fail. LangGraph error handling strategies:

1. **Retry** — try the same node again (transient failures)
2. **Fallback** — route to alternative node
3. **Graceful degradation** — skip failed check, proceed with partial data
4. **Circuit breaker** — after N failures, stop trying that path

In [ ]:
import random

class ResilientState(TypedDict):
    transaction_id: str
    fraud_result: Optional[Dict]
    retry_count: int
    max_retries: int
    error_log: Annotated[List[str], operator.add]
    evidence: Annotated[List[str], operator.add]


def unreliable_fraud_check(state: ResilientState) -> dict:
    """Simulates an unreliable external fraud ML service."""
    # 60% failure rate for demo
    if random.random() < 0.6:
        retry = state.get("retry_count", 0) + 1
        return {
            "retry_count": retry,
            "error_log": [f"Attempt {retry}: ML service timeout (500ms)"],
        }

    return {
        "fraud_result": {"score": 85, "signals": ["velocity", "geo_mismatch"]},
        "evidence": ["Fraud ML service returned: score=85"],
    }


def fallback_rule_engine(state: ResilientState) -> dict:
    """Fallback: simple rule-based fraud check when ML service is down."""
    txn = NOVAPAY_TRANSACTIONS.get(state["transaction_id"], {})
    score = 50  # Conservative base
    if txn.get("velocity_flag"):
        score += 20
    if txn.get("amount", 0) > 3000:
        score += 15

    return {
        "fraud_result": {"score": score, "signals": ["rule_based_fallback"], "degraded": True},
        "evidence": [f"Fallback rule engine: score={score} (ML service unavailable after {state.get('retry_count', 0)} retries)"],
    }


def should_retry(state: ResilientState) -> str:
    """Decide: retry, fallback, or done."""
    if state.get("fraud_result") is not None:
        return "done"
    if state.get("retry_count", 0) >= state.get("max_retries", 3):
        return "fallback"
    return "retry"


# Build resilient graph
resilient_graph = StateGraph(ResilientState)
resilient_graph.add_node("fraud_check", unreliable_fraud_check)
resilient_graph.add_node("fallback", fallback_rule_engine)

resilient_graph.add_edge(START, "fraud_check")
resilient_graph.add_conditional_edges(
    "fraud_check",
    should_retry,
    {"retry": "fraud_check", "fallback": "fallback", "done": END},
)
resilient_graph.add_edge("fallback", END)

resilient_app = resilient_graph.compile()

# Run it
random.seed(42)  # For reproducibility
result = resilient_app.invoke({
    "transaction_id": "TXN-101",
    "fraud_result": None,
    "retry_count": 0,
    "max_retries": 3,
    "error_log": [],
    "evidence": [],
})

print("📊 Resilient Execution Result:")
print(f"  Fraud Result: {result['fraud_result']}")
print(f"  Retries used: {result['retry_count']}")
print(f"  Error log: {result['error_log']}")
print(f"  Evidence: {result['evidence']}")

## Section 8: Production Deployment Patterns

How do you deploy a LangGraph agent in production?

In [ ]:
display(HTML("""
<table style=\'border-collapse: collapse; width: 100%; font-size: 14px;\'>
<tr style=\'background: #2c3e50; color: white;\'>
    <th style=\'padding: 12px; border: 1px solid #ddd;\'>Option</th>
    <th style=\'padding: 12px; border: 1px solid #ddd;\'>How</th>
    <th style=\'padding: 12px; border: 1px solid #ddd;\'>Pros</th>
    <th style=\'padding: 12px; border: 1px solid #ddd;\'>Cons</th>
</tr>
<tr><td style=\'padding: 10px; border: 1px solid #ddd;\'><b>LangGraph Platform (Cloud)</b></td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>LangSmith-hosted, managed infra</td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Zero ops, built-in persistence, streaming</td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Vendor lock-in, data leaves your VPC</td></tr>
<tr><td style=\'padding: 10px; border: 1px solid #ddd;\'><b>Self-hosted (ECS/EKS)</b></td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Docker + PostgreSQL + your infra</td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Full control, data in your VPC</td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>You manage scaling, checkpointing infra</td></tr>
<tr><td style=\'padding: 10px; border: 1px solid #ddd;\'><b>AWS Lambda</b></td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Serverless execution per investigation</td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Cost-efficient for bursty traffic</td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Cold starts, 15min timeout, no streaming</td></tr>
<tr><td style=\'padding: 10px; border: 1px solid #ddd;\'><b>AgentCore (via Strands)</b></td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Rewrite agent in Strands, deploy to AgentCore</td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Managed runtime, Gateway, Identity, AWS-native</td>
    <td style=\'padding: 10px; border: 1px solid #ddd;\'>Requires migration, less graph control</td></tr>
</table>

<h3 style=\'margin-top: 20px;\'>🎯 NovaPay\'s Choice: Self-hosted for now, migration to AgentCore planned</h3>
<p>Reason: Data sovereignty requirements (PCI DSS) prevent cloud-hosted LangGraph Platform. AgentCore migration is in progress (see Lab 4: Migration to AgentCore).</p>
"""))

## Section 9: LangGraph vs Strands for Production

Detailed comparison using NovaPay's actual requirements:

In [ ]:
display(HTML("""
<table style=\'border-collapse: collapse; width: 100%; font-size: 13px;\'>
<tr style=\'background: #8e44ad; color: white;\'>
    <th style=\'padding: 10px; border: 1px solid #ddd;\'>NovaPay Requirement</th>
    <th style=\'padding: 10px; border: 1px solid #ddd;\'>LangGraph</th>
    <th style=\'padding: 10px; border: 1px solid #ddd;\'>Strands Agents</th>
</tr>
<tr><td style=\'padding: 8px; border: 1px solid #ddd;\'>Parallel fraud+compliance checks</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Native fan-out/fan-in</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>asyncio.gather() manually</td></tr>
<tr><td style=\'padding: 8px; border: 1px solid #ddd;\'>Human-in-the-loop</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>interrupt_before/after</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>HITL tool pattern</td></tr>
<tr><td style=\'padding: 8px; border: 1px solid #ddd;\'>State persistence</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Checkpointer (Postgres/SQLite)</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>AgentCore Memory</td></tr>
<tr><td style=\'padding: 8px; border: 1px solid #ddd;\'>Streaming for UI</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>stream() + astream_events()</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Event loop streaming</td></tr>
<tr><td style=\'padding: 8px; border: 1px solid #ddd;\'>Error recovery</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Conditional retry edges</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Try/except in tools</td></tr>
<tr><td style=\'padding: 8px; border: 1px solid #ddd;\'>Visualization</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Mermaid diagrams, LangSmith traces</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Manual logging</td></tr>
<tr><td style=\'padding: 8px; border: 1px solid #ddd;\'>Managed deployment</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>LangGraph Platform (3rd party)</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>AgentCore (AWS-native)</td></tr>
<tr><td style=\'padding: 8px; border: 1px solid #ddd;\'>Cost at 50K investigations/day</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Self-managed infra cost</td>
    <td style=\'padding: 8px; border: 1px solid #ddd;\'>Pay-per-use AgentCore</td></tr>
</table>

<h3 style=\'margin-top: 16px;\'>🎯 Bottom Line for NovaPay:</h3>
<ul>
<li><b>Use LangGraph</b> when you need explicit control over execution topology (parallel branches, complex routing, subgraphs)</li>
<li><b>Use Strands</b> when you want simpler code + managed AWS deployment</li>
<li><b>Best of both:</b> Design with LangGraph for complex flows, deploy simple agents via Strands/AgentCore</li>
</ul>
"""))

## 🧠 Knowledge Check

**Q1:** Why does NovaPay use parallel branches instead of sequential execution?

**Q2:** What happens if the system crashes while waiting for human approval at the interrupt?

**Q3:** When should you use a subgraph vs a single node?

**Q4:** What's the trade-off of `stream_mode="updates"` vs `stream_mode="values"`?

**Q5:** How would you add a circuit breaker to the retry pattern?

---

## ✅ Lab Complete

You've built advanced LangGraph patterns for NovaPay:

- Parallel branch execution (fan-out/fan-in)
- Subgraphs for modular investigation logic
- Human-in-the-loop with interrupt/resume
- Persistent checkpointing
- Streaming for real-time UI
- Error handling with retry and graceful degradation
- LangGraph vs Strands production comparison